In [ ]:
# Cell 0 — Setup
# Load ttf_curve.csv (or synthetic fallback), define parser + validity helpers.
import sys, os, warnings, re, calendar
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.colors as pc
from pathlib import Path
warnings.filterwarnings('ignore')

# ── Locate project root ────────────────────────────────────────────────────────
for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c))
        os.chdir(_c)
        print(f'Project root: {_c}')
        break

# ── Load TTF forward curve ─────────────────────────────────────────────────────
RAW     = Path('data/raw')
TTF_CSV = RAW / 'ttf_curve.csv'

if TTF_CSV.exists():
    ttf = pd.read_csv(TTF_CSV, index_col=0, parse_dates=True)
    ttf.index = pd.to_datetime(ttf.index).tz_localize(None)
    ttf = ttf.sort_index()
    _src = 'Databento NDEX.IMPACT'
    print(f'✅ Loaded real data: data/raw/ttf_curve.csv ({len(ttf)} rows)')
    print(f'   Date range : {ttf.index[0].date()} → {ttf.index[-1].date()}')
    print(f'   Columns    : {list(ttf.columns[:6])} ... (total {len(ttf.columns)})')
else:
    print('⚠️ Using synthetic fallback — real CSV not found at data/raw/ttf_curve.csv')
    print('   Run: python scripts/fetch_ttf_curve.py --api-key <KEY>  to use real data')
    np.random.seed(42)
    _dates = pd.bdate_range('2019-01-02', '2026-06-18')
    _d = {}
    for _m in range(1, 25):
        _base = np.where(
            _dates.year < 2022, 25.0,
            np.where(_dates.year == 2022, 95.0,
            np.where(_dates.year == 2023, 55.0, 33.0))
        ).astype(float)
        _seas  = 1.0 + 0.06 * np.cos(2 * np.pi * (_dates.month - 1) / 12 + np.pi)
        _slope = 1.0 + 0.003 * _m
        _walk  = np.cumsum(np.random.normal(0, 0.25, len(_dates)))
        _d[f'M{_m}'] = np.clip(_base * _seas * _slope + _walk, 5, 300).round(3)
    ttf = pd.DataFrame(_d, index=_dates)
    ttf.index.name = 'date'
    _src = 'SYNTHETIC — replace with real data'
    print(f'   Synthetic TTF: {ttf.shape}  {ttf.index[0].date()} → {ttf.index[-1].date()}')

# ── Month-name lookup (abbr + full, case-insensitive) ─────────────────────────
_MONTH_MAP: dict = {}
for _i in range(1, 13):
    _MONTH_MAP[calendar.month_abbr[_i].lower()] = _i
    _MONTH_MAP[calendar.month_name[_i].lower()]  = _i


# ── Target period parser ───────────────────────────────────────────────────────
def parse_target(s: str) -> list:
    """
    Parse a delivery-period string into a list of (delivery_year, delivery_month) tuples.

    Supported formats
    -----------------
    'Mar 2027'   -> [(2027, 3)]
    'Q3 2027'    -> [(2027, 7), (2027, 8), (2027, 9)]
    'Summer 26'  -> [(2026, 4) ... (2026, 9)]  Apr-Sep
    'Winter 26'  -> [(2026,10) ... (2027, 3)]  Oct-Mar
    'Cal 2026'   -> [(2026, 1) ... (2026,12)]
    """
    s = s.strip()

    # Cal / CAL YYYY
    m = re.match(r'^(?:Cal|CAL)\s+(\d{4})$', s, re.I)
    if m:
        yr = int(m.group(1))
        return [(yr, mo) for mo in range(1, 13)]

    # Q[1-4] YYYY
    m = re.match(r'^Q([1-4])\s+(\d{4})$', s, re.I)
    if m:
        q, yr = int(m.group(1)), int(m.group(2))
        start = (q - 1) * 3 + 1
        return [(yr, mo) for mo in range(start, start + 3)]

    # Summer YY or YYYY  ->  Apr-Sep
    m = re.match(r'^Summer\s+(\d{2,4})$', s, re.I)
    if m:
        raw = m.group(1)
        yr  = int(raw) if len(raw) == 4 else 2000 + int(raw)
        return [(yr, mo) for mo in range(4, 10)]

    # Winter YY or YYYY  ->  Oct(yr) through Mar(yr+1)
    m = re.match(r'^Winter\s+(\d{2,4})$', s, re.I)
    if m:
        raw = m.group(1)
        yr  = int(raw) if len(raw) == 4 else 2000 + int(raw)
        return [(yr, 10), (yr, 11), (yr, 12), (yr + 1, 1), (yr + 1, 2), (yr + 1, 3)]

    # Month YYYY  (abbreviated or full name)
    m = re.match(r'^(\w+)\s+(\d{4})$', s)
    if m:
        mo_str, yr = m.group(1).lower(), int(m.group(2))
        if mo_str in _MONTH_MAP:
            return [(yr, _MONTH_MAP[mo_str])]

    raise ValueError(
        f'Cannot parse {s!r}.  Supported: '
        "'Mar 2027', 'Q3 2027', 'Summer 26', 'Winter 26', 'Cal 2026'"
    )


def delivery_window_label(delivery_months: list) -> str:
    """Human-readable delivery window, e.g. 'Jan 2026 – Dec 2026'."""
    if not delivery_months:
        return '?'
    (y0, m0), (y1, m1) = delivery_months[0], delivery_months[-1]
    s0 = f'{calendar.month_abbr[m0]} {y0}'
    s1 = f'{calendar.month_abbr[m1]} {y1}'
    return s0 if s0 == s1 else f'{s0} – {s1}'


# ── Core validity check ────────────────────────────────────────────────────────
def is_valid_trade_date(trade_dt: pd.Timestamp, delivery_months: list) -> bool:
    """
    True iff ALL delivery months satisfy  1 <= months_ahead <= 24.

    months_ahead = (del_yr - trade_yr)*12 + (del_mo - trade_mo)

    Once months_ahead < 1 for ANY delivery month (delivery has started for that
    leg), this trade date is excluded.  ICE Endex TTF futures stop trading ~2
    business days before the delivery month starts, so the last valid settlement
    for the front leg falls in the last few days of the month before delivery.
    """
    for del_yr, del_mo in delivery_months:
        ma = (del_yr - trade_dt.year) * 12 + (del_mo - trade_dt.month)
        if not (1 <= ma <= 24):
            return False
    return True


# ── Build price series ─────────────────────────────────────────────────────────
def build_price_series(ttf_df: pd.DataFrame, delivery_months: list) -> pd.Series:
    """
    Return pd.Series(trade_date -> average implied price) for a delivery period.

    For each trade date:
      1. Compute months_ahead for every delivery month in the target period.
      2. Include ONLY if ALL months satisfy  1 <= months_ahead <= 24.
      3. Price = average of M{months_ahead} columns for all delivery months.
      4. Exclude if any M{months_ahead} column is NaN.

    The series terminates naturally when the first delivery month's months_ahead
    drops to 0 (delivery has started — the forward contract no longer exists).
    """
    rows = []
    for trade_dt, row in ttf_df.iterrows():
        if not is_valid_trade_date(trade_dt, delivery_months):
            continue
        prices = []
        ok = True
        for del_yr, del_mo in delivery_months:
            ma  = (del_yr - trade_dt.year) * 12 + (del_mo - trade_dt.month)
            col = f'M{ma}'
            if col not in row.index or pd.isna(row[col]):
                ok = False
                break
            prices.append(float(row[col]))
        if ok and prices:
            rows.append({'date': trade_dt, 'price': float(np.mean(prices))})
    if not rows:
        return pd.Series(dtype=float, name='price')
    return pd.DataFrame(rows).set_index('date')['price'].sort_index()


# ── Parser smoke-test ──────────────────────────────────────────────────────────
print('Parser check:')
for _tc in ['Mar 2027', 'Q3 2027', 'Summer 26', 'Winter 26', 'Cal 2026']:
    _dm = parse_target(_tc)
    print(f'  {_tc:<14} -> {len(_dm):>2} month(s)  [{delivery_window_label(_dm)}]')

# ═══════════════════════════════════════════════════════════════════════════════
# EXPLICIT VALIDITY AUDIT — Cal 2026
# Confirm: NO trade dates in the series fall on or after 2026-01-01.
# Binding constraint: Jan 2026 needs months_ahead >= 1, which fails for any
# trade date in Jan 2026+ (months_ahead = 0 when trade month == delivery month).
# ═══════════════════════════════════════════════════════════════════════════════
print()
print('─'*55)
print('VALIDITY AUDIT: Cal 2026  (delivery Jan 2026 – Dec 2026)')
print('─'*55)
_cal26_months = parse_target('Cal 2026')
_cal26_series = build_price_series(ttf, _cal26_months)

if _cal26_series.empty:
    print('  (no data — ttf_curve.csv absent or does not cover late 2025)')
else:
    _last  = _cal26_series.index[-1]
    _first = _cal26_series.index[0]
    _bad   = _cal26_series[_cal26_series.index >= '2026-01-01']
    print(f'  First quoted date         : {_first.date()}')
    print(f'  Last quoted date          : {_last.date()}')
    print(f'  Expected: last date must be in Dec 2025 (months_ahead=1 for Jan 2026)')
    print(f'  Trade dates >= 2026-01-01 : {len(_bad)}  (must be 0)')
    if len(_bad) == 0:
        print(f'  ✅ PASS — series correctly stops before Jan 2026 delivery begins')
    else:
        print(f'  ❌ FAIL — {len(_bad)} invalid dates found:')
        print('     ', [str(d.date()) for d in _bad.index[:5]])
print('─'*55)

In [ ]:
# ── EDIT THIS LINE to track a different contract ──────────────────────────────
TARGET = 'Cal 2026'   # 'Mar 2027' | 'Q3 2027' | 'Summer 26' | 'Winter 26' | 'Cal 2026'
# ─────────────────────────────────────────────────────────────────────────────

delivery_months = parse_target(TARGET)
win_label       = delivery_window_label(delivery_months)
price_series    = build_price_series(ttf, delivery_months)

if price_series.empty:
    print(f'No quotable dates for {TARGET!r}  (delivery: {win_label})')
    print(f'TTF data spans {ttf.index[0].date()} \u2192 {ttf.index[-1].date()}')
    print('Possible reasons:')
    print('  - Delivery already started/finished (series terminates at delivery start)')
    print('  - All delivery months are >24 months out (outside M1-M24 curve depth)')
    print('  - TTF data does not cover the quotable window for this contract')
else:
    first_dt = price_series.index[0]
    last_dt  = price_series.index[-1]
    p_first  = float(price_series.iloc[0])
    p_last   = float(price_series.iloc[-1])
    p_min    = float(price_series.min())
    p_max    = float(price_series.max())
    idx_min  = price_series.idxmin()
    idx_max  = price_series.idxmax()

    pad     = (p_max - p_min) * 0.07
    y_range = [p_min - pad, p_max + pad]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=price_series.index, y=price_series.values,
        mode='lines',
        line=dict(color='#2E75B6', width=1.8),
        name=TARGET,
        hovertemplate='%{x|%d %b %Y}: \u20ac%{y:.2f}/MWh<extra></extra>',
    ))

    # Mark the delivery wall: last valid trade date
    fig.add_vline(
        x=last_dt,
        line_dash='dot', line_color='#E74C3C', line_width=1.5,
        annotation_text=(
            f'Last valid: {last_dt.strftime("%d %b %Y")}<br>'
            'Delivery starts \u2192 no forward price'
        ),
        annotation_position='top left',
        annotation_font=dict(size=10, color='#E74C3C'),
    )

    # Annotate first, min, max (deduplicated by position)
    _ann_data = [
        (first_dt, p_first, f'First: {p_first:.2f}', '#555',    40, -28),
        (idx_min,  p_min,   f'Low {p_min:.2f}',       '#27AE60',  0,  35),
        (idx_max,  p_max,   f'High {p_max:.2f}',      '#C0392B',  0, -35),
    ]
    _seen = set()
    for _dt, _px, _lbl, _col, _ax, _ay in _ann_data:
        _key = (_dt, round(_px, 1))
        if _key not in _seen:
            _seen.add(_key)
            fig.add_annotation(
                x=_dt, y=_px, text=_lbl,
                showarrow=True, arrowhead=2, ax=_ax, ay=_ay,
                font=dict(color=_col, size=10),
            )

    fig.update_layout(
        title=(
            f'{TARGET}  (delivery {win_label}) \u2014 forward price history, '
            f'valid through ~{last_dt.strftime("%b %Y")}<br>'
            '<sub>Series stops when delivery begins, not from a data gap. '
            'Red dashed line = last valid forward settlement.</sub>'
        ),
        xaxis_title='Trade Date',
        yaxis=dict(title='EUR/MWh', range=y_range),
        height=480,
        template='plotly_white',
        hovermode='x unified',
    )
    fig.show()

    print(f'\n{TARGET}: {first_dt.date()} \u2192 {last_dt.date()}  ({len(price_series)} trading days)')
    print(f'  Last valid trade date : {last_dt.date()}')
    print(f'  Annotated last-valid-date: {last_dt.date()}')
    print(f'  Series stops because delivery of [{win_label}] begins \u2014 not a data gap.')

In [ ]:
# Cell 2 — Statistics
# Summary stats for the single TARGET defined in Cell 1.

if price_series.empty:
    print(f'No data for {TARGET}.')
else:
    ps = price_series.dropna()

    log_ret  = np.log(ps / ps.shift(1)).dropna()
    ann_vol  = float(log_ret.std() * np.sqrt(252)) * 100
    roll_max = ps.cummax()
    max_dd   = float(((ps - roll_max) / roll_max * 100).min())
    pct_chg  = (float(ps.iloc[-1]) / float(ps.iloc[0]) - 1) * 100

    first_del = delivery_months[0]

    print(f'{"="*62}')
    print(f'  Contract             : {TARGET}')
    print(f'  Delivery window      : {win_label}')
    print(f'  Quotable period      : {ps.index[0].date()}  \u2192  {ps.index[-1].date()}')
    print(f'  Trading days quoted  : {len(ps):,}')
    print()
    print(f'  First quoted price   : \u20ac{ps.iloc[0]:.3f}/MWh  ({ps.index[0].date()})')
    print(f'  Last quoted price    : \u20ac{ps.iloc[-1]:.3f}/MWh  ({ps.index[-1].date()})')
    print(f'  % change             : {pct_chg:+.1f}%')
    print(f'  Min                  : \u20ac{ps.min():.3f}/MWh  ({ps.idxmin().date()})')
    print(f'  Max                  : \u20ac{ps.max():.3f}/MWh  ({ps.idxmax().date()})')
    print(f'  Realized vol (p.a.)  : {ann_vol:.1f}%')
    print(f'  Max drawdown         : {max_dd:.1f}%')
    print(f'{"="*62}')
    print()
    print(f'  \u2190 Series terminates on {ps.index[-1].date()} because delivery of {TARGET}')
    print(f'    begins {calendar.month_abbr[first_del[1]]} {first_del[0]}.')
    print(f'    ICE Endex TTF futures stop trading ~2 business days before the')
    print(f'    delivery month starts. Once months_ahead < 1 for any delivery leg,')
    print(f'    the contract no longer exists as a tradeable forward instrument.')

In [ ]:
# ── EDIT THIS LIST to compare different contracts ─────────────────────────────
TARGETS = ['Mar 2027', 'Summer 26', 'Cal 2026']   # any mix of parse_target formats
# ─────────────────────────────────────────────────────────────────────────────
# Each line stops at its own delivery start — lines naturally end at different
# x-axis positions.  Legend shows [delivery window] and last valid month so you
# can audit the cutoff date at a glance.

COLORS   = pc.qualitative.Set1 + pc.qualitative.Set2
fig_norm = go.Figure()

print(f'Building series for {len(TARGETS)} contracts \u2014 each stops at its own delivery start')
print()

for i, tgt in enumerate(TARGETS):
    try:
        dm = parse_target(tgt)
    except ValueError as e:
        print(f'  Skipped {tgt!r}: {e}')
        continue

    ps = build_price_series(ttf, dm)
    if ps.empty:
        print(f'  {tgt:<18} : no quotable dates in TTF data range \u2014 skipped')
        continue

    ps   = ps.dropna()
    p0   = float(ps.iloc[0])
    ps_n = (ps / p0 - 1) * 100
    win  = delivery_window_label(dm)

    # Legend: contract name + delivery window + when the series ends
    legend = f'{tgt}  [{win}]  last={ps.index[-1].strftime("%b %Y")}'
    color  = COLORS[i % len(COLORS)]

    fig_norm.add_trace(go.Scatter(
        x=ps_n.index, y=ps_n.values,
        name=legend, mode='lines',
        line=dict(color=color, width=1.8),
        hovertemplate=f'<b>{tgt}</b><br>%{{x|%d %b %Y}}: %{{y:+.1f}}%<extra></extra>',
    ))

    print(
        f'  {tgt:<18} : {len(ps):>4} days  '
        f'{ps.index[0].date()} \u2192 {ps.index[-1].date()}  '
        f'[{win}]  '
        f'first=\u20ac{p0:.2f}  last=\u20ac{ps.iloc[-1]:.2f}'
    )

fig_norm.add_hline(y=0, line_dash='dot', line_color='black', line_width=1)
fig_norm.update_layout(
    title=(
        'Contract Forward Prices \u2014 % change from first observation<br>'
        '<sub>Each line stops at its own delivery start. '
        'Lines naturally end at different dates. '
        'Legend shows [delivery window] and last valid month.</sub>'
    ),
    xaxis_title='Trade Date',
    yaxis_title='% change from first quote',
    height=520,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        orientation='h',
        y=-0.28, x=0, xanchor='left',
        font=dict(size=10),
    ),
)
fig_norm.show()

print()
print('Note: lines do NOT share the same x-axis end point.')
print('Each contract has its own delivery-start cutoff \u2014 this is correct by design.')